<a href="https://colab.research.google.com/github/krishkundu00033-netizen/python_project/blob/main/mobile_usage.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# -------------------------------------------------------------------------
    # Task 1: Load and inspect the app usage dataset
    # -------------------------------------------------------------------------
    print("=" * 72)
    print("1. DATASET OVERVIEW & INSPECTION")
    print("=" * 72)
    df = pd.read_csv(filepath)
    print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
    print("\nFirst 5 rows:")
    print(df.head())
    print("\nData types & non-null counts:")
    print(df.info())

    session_id_col = resolve_column(df, ["Session_ID", "session_id"])
    user_id_col = resolve_column(df, ["User_ID", "user_id"])
    date_col = resolve_column(df, ["Session_Date", "Session_Timestamp", "Timestamp", "Date"])
    duration_col = resolve_column(df, ["Session_Duration_Mins", "Duration_Mins", "Session_Duration", "Duration"])
    rating_col = resolve_column(df, ["User_Rating", "Rating"], required=False)
    feature_col = resolve_column(df, ["Feature_Used", "Feature", "App_Feature"])
    device_col = resolve_column(df, ["Device_Type", "Device", "Platform"])
    age_col = resolve_column(df, ["Age_Group", "User_Age_Group", "Age_Bracket", "Age"], required=False)

1. DATASET OVERVIEW & INSPECTION
Shape: 1212 rows x 8 columns

First 5 rows:
  Session_ID User_ID         Session_Date  Session_Duration_Mins  \
0     S00834   U0051  2026-04-23 09:51:55                   13.1   
1     S00301   U0108  2026-07-08 19:42:39                    1.0   
2     S01039   U0079  2026-03-11 20:45:18                    3.4   
3     S00840   U0163  2026-07-30 16:40:27                   25.3   
4     S00236   U0116  2026-08-10 22:58:06                   32.2   

   Feature_Used Device_Type Age_Group  User_Rating  
0     Home Feed         iOS     18-24          3.0  
1       Profile     Android     18-24          3.0  
2     Messaging     Android       45+          2.0  
3     Messaging     Android     35-44          4.0  
4  Video Player     Android       45+          4.0  

Data types & non-null counts:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1212 entries, 0 to 1211
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype  
---  --

In [4]:
# -------------------------------------------------------------------------
# Task 2: Check for missing values and duplicate Session_ID records
# -------------------------------------------------------------------------
print("\n" + "=" * 72)
print("2. MISSING VALUES & DUPLICATE SESSION_ID CHECK")
print("=" * 72)

# Resolve session_id_col in the global scope to ensure this cell runs independently
session_id_col = resolve_column(df, ["Session_ID", "session_id"])

missing_counts = df.isna().sum()
missing_pct = (df.isna().mean() * 100).round(2)
missing_report = pd.DataFrame({"missing_count": missing_counts, "missing_pct": missing_pct})
print(missing_report[missing_report["missing_count"] > 0])
if missing_report["missing_count"].sum() == 0:
    print("No missing values found across any column.")

dup_mask = df.duplicated(subset=[session_id_col], keep="first")
num_dups = int(dup_mask.sum())
print(f"\nDuplicate {session_id_col} rows found: {num_dups}")
if num_dups > 0:
    df = df.loc[~dup_mask].copy()
    print(f"Removed {num_dups} duplicate session(s). Remaining rows: {len(df)}")


2. MISSING VALUES & DUPLICATE SESSION_ID CHECK
                       missing_count  missing_pct
Session_Duration_Mins             46         3.80
User_Rating                       90         7.43

Duplicate Session_ID rows found: 12
Removed 12 duplicate session(s). Remaining rows: 1200


In [5]:
# -------------------------------------------------------------------------
# Task 3: Handle missing Session_Duration_Mins or User_Rating values
# -------------------------------------------------------------------------
print("\n" + "=" * 72)
print("3. HANDLING MISSING DURATION & RATING VALUES")
print("=" * 72)
# Justification:
# - Session duration is right-skewed in study apps (occasional long cramming sessions),
#   so per-feature median (falling back to overall median) preserves feature differences
#   without letting outliers distort imputed values.
# - User ratings are ordinal/bounded (typically 1-5); if missing, we keep NaN when
#   computing mean ratings so unrated sessions don't artificially pull ratings toward a guess,
#   or impute overall median if a complete numeric column is needed for modeling.
df[duration_col] = pd.to_numeric(df[duration_col], errors="coerce")
missing_dur_before = int(df[duration_col].isna().sum())
feature_medians = df.groupby(feature_col)[duration_col].transform("median")
overall_median_dur = float(df[duration_col].median())
df[duration_col] = df[duration_col].fillna(feature_medians).fillna(overall_median_dur)
print(
    f"Imputed {missing_dur_before} missing {duration_col} value(s) using feature-level median "
    f"(fallback overall median: {overall_median_dur:.2f} mins)."
)

if rating_col is not None:
    df[rating_col] = pd.to_numeric(df[rating_col], errors="coerce")
    missing_rat_before = int(df[rating_col].isna().sum())
    print(
        f"Found {missing_rat_before} missing {rating_col} value(s); retained as NaN for unbiased "
        f"mean rating calculation (mean() ignores NaN by default)."
    )


3. HANDLING MISSING DURATION & RATING VALUES
Imputed 45 missing Session_Duration_Mins value(s) using feature-level median (fallback overall median: 6.50 mins).
Found 90 missing User_Rating value(s); retained as NaN for unbiased mean rating calculation (mean() ignores NaN by default).


In [6]:
# -------------------------------------------------------------------------
# Task 4: Convert Session_Date to datetime and extract month, weekday, hour
# -------------------------------------------------------------------------
print("\n" + "=" * 72)
print("4. DATETIME EXTRACTION (MONTH, WEEKDAY, HOUR)")
print("=" * 72)
df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
df["Month"] = df[date_col].dt.month
df["Month_Name"] = df[date_col].dt.month_name()
df["Weekday"] = df[date_col].dt.day_name()
df["Hour"] = df[date_col].dt.hour
print(df[[date_col, "Month", "Month_Name", "Weekday", "Hour"]].head())


4. DATETIME EXTRACTION (MONTH, WEEKDAY, HOUR)
         Session_Date  Month Month_Name    Weekday  Hour
0 2026-04-23 09:51:55      4      April   Thursday     9
1 2026-07-08 19:42:39      7       July  Wednesday    19
2 2026-03-11 20:45:18      3      March  Wednesday    20
3 2026-07-30 16:40:27      7       July   Thursday    16
4 2026-08-10 22:58:06      8     August     Monday    22


In [7]:
# -------------------------------------------------------------------------
# Task 5: Total sessions, unique users, and average session duration
# -------------------------------------------------------------------------
print("\n" + "=" * 72)
print("5. CORE ENGAGEMENT SUMMARY")
print("=" * 72)
total_sessions = len(df)
unique_users_count = df[user_id_col].nunique()
avg_session_duration = float(df[duration_col].mean())
print(f"Total Sessions:          {total_sessions:,}")
print(f"Unique Users:            {unique_users_count:,}")
print(f"Average Session Duration:{avg_session_duration:.2f} mins")


5. CORE ENGAGEMENT SUMMARY
Total Sessions:          1,200
Unique Users:            335
Average Session Duration:11.04 mins


In [8]:
# -------------------------------------------------------------------------
# Task 6: Top 10 most frequently used app features
# -------------------------------------------------------------------------
print("\n" + "=" * 72)
print("6. TOP 10 MOST FREQUENTLY USED FEATURES")
print("=" * 72)
feature_counts = df[feature_col].value_counts()
top_10_features_df = pd.DataFrame({
    "Feature": feature_counts.head(10).index,
    "Session_Count": feature_counts.head(10).values,
    "Share_Pct": (feature_counts.head(10).values / total_sessions * 100).round(2),
})
print(top_10_features_df.to_string(index=False))


6. TOP 10 MOST FREQUENTLY USED FEATURES
      Feature  Session_Count  Share_Pct
    Home Feed            243      20.25
    Messaging            176      14.67
       Search            170      14.17
 Video Player            146      12.17
Notifications            108       9.00
      Profile            102       8.50
     Settings             56       4.67
    Quiz Mode             54       4.50
     Payments             54       4.50
   Flashcards             34       2.83


In [9]:
# -------------------------------------------------------------------------
# Task 7: Compare average session duration across device types
# -------------------------------------------------------------------------
print("\n" + "=" * 72)
print("7. AVERAGE SESSION DURATION BY DEVICE TYPE")
print("=" * 72)
device_duration = (
    df.groupby(device_col)[duration_col]
    .agg(sessions="count", mean_mins="mean", median_mins="median")
    .sort_values("mean_mins", ascending=False)
    .round(2)
)
print(device_duration)


7. AVERAGE SESSION DURATION BY DEVICE TYPE
             sessions  mean_mins  median_mins
Device_Type                                  
Web                92      13.61         7.35
Tablet            129      12.14         7.50
iOS               393      11.80         7.20
Android           586       9.89         5.50


In [10]:
# -------------------------------------------------------------------------
# Task 8: Rank user age groups by total number of sessions
# -------------------------------------------------------------------------
print("\n" + "=" * 72)
print("8. AGE GROUPS RANKED BY TOTAL SESSIONS")
print("=" * 72)
if age_col is not None:
    age_ranking = (
        df.groupby(age_col, dropna=False)[session_id_col]
        .count()
        .reset_index(name="Total_Sessions")
        .sort_values("Total_Sessions", ascending=False)
        .reset_index(drop=True)
    )
    age_ranking["Rank"] = age_ranking.index + 1
    age_ranking["Share_Pct"] = (age_ranking["Total_Sessions"] / total_sessions * 100).round(2)
    print(age_ranking[["Rank", age_col, "Total_Sessions", "Share_Pct"]].to_string(index=False))
else:
    print("No age or age-group column found in dataset.")


8. AGE GROUPS RANKED BY TOTAL SESSIONS
 Rank Age_Group  Total_Sessions  Share_Pct
    1     18-24             415      34.58
    2     25-34             325      27.08
    3     35-44             186      15.50
    4       45+             164      13.67
    5     13-17             110       9.17


In [11]:
# -------------------------------------------------------------------------
# Task 9: Compare average user ratings across app features
# -------------------------------------------------------------------------
print("\n" + "=" * 72)
print("9. AVERAGE USER RATINGS ACROSS APP FEATURES")
print("=" * 72)
if rating_col is not None:
    feature_ratings = (
        df.groupby(feature_col)[rating_col]
        .agg(rated_sessions="count", avg_rating="mean", median_rating="median")
        .sort_values("avg_rating", ascending=False)
        .round(2)
    )
    print(feature_ratings)
else:
    print("No user rating column found in dataset.")


9. AVERAGE USER RATINGS ACROSS APP FEATURES
                  rated_sessions  avg_rating  median_rating
Feature_Used                                               
Video Player                 135        4.21            4.0
Quiz Mode                     54        4.20            4.0
Flashcards                    32        4.19            4.0
Messaging                    158        4.03            4.0
Progress Tracker              24        3.92            4.0
Offline Mode                  30        3.87            4.0
Home Feed                    225        3.80            4.0
Profile                       99        3.75            4.0
Search                       154        3.45            3.0
Settings                      51        3.24            3.0
Notifications                 97        3.12            3.0
Payments                      51        3.06            3.0


In [12]:
# -------------------------------------------------------------------------
# Task 10: Identify peak app usage hours and busiest weekdays
# -------------------------------------------------------------------------
print("\n" + "=" * 72)
print("10. PEAK USAGE HOURS & BUSIEST WEEKDAYS")
print("=" * 72)
hourly_usage = df.groupby("Hour")[session_id_col].count().sort_values(ascending=False)
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekday_usage = (
    df.groupby("Weekday")[session_id_col]
    .count()
    .reindex(weekday_order)
    .fillna(0)
    .astype(int)
    .sort_values(ascending=False)
)
print("Top 5 Peak Hours (24h format):")
print(hourly_usage.head(5).to_string())
print("\nWeekdays Ranked by Session Volume:")
print(weekday_usage.to_string())


10. PEAK USAGE HOURS & BUSIEST WEEKDAYS
Top 5 Peak Hours (24h format):
Hour
20    132
19    117
18     87
21     86
17     73

Weekdays Ranked by Session Volume:
Weekday
Friday       206
Saturday     203
Thursday     182
Sunday       162
Wednesday    160
Tuesday      148
Monday       139


In [13]:
# -------------------------------------------------------------------------
# Task 11: List of most-used features & tuple of fixed device categories
# -------------------------------------------------------------------------
print("\n" + "=" * 72)
print("11. PYTHON LIST (TOP FEATURES) & TUPLE (DEVICE CATEGORIES)")
print("=" * 72)
most_used_features: list[str] = feature_counts.head(10).index.tolist()
fixed_device_categories: tuple[str, ...] = tuple(sorted(df[device_col].dropna().unique().tolist()))
print("Most-used features (list):", most_used_features)
print("Fixed device categories (tuple):", fixed_device_categories)


11. PYTHON LIST (TOP FEATURES) & TUPLE (DEVICE CATEGORIES)
Most-used features (list): ['Home Feed', 'Messaging', 'Search', 'Video Player', 'Notifications', 'Profile', 'Settings', 'Quiz Mode', 'Payments', 'Flashcards']
Fixed device categories (tuple): ('Android', 'Tablet', 'Web', 'iOS')


In [14]:
# -------------------------------------------------------------------------
# Task 12: Set of unique users & function for average feature duration
# -------------------------------------------------------------------------
print("\n" + "=" * 72)
print("12. UNIQUE USERS SET & FEATURE DURATION FUNCTION")
print("=" * 72)
unique_users_set: set = set(df[user_id_col].dropna().unique().tolist())
print(f"Unique users in set: {len(unique_users_set)}")
if most_used_features:
    sample_feature = most_used_features[0]
    sample_avg = avg_duration_for_feature(df, feature_col, duration_col, sample_feature)
    print(f"Average duration for {sample_feature!r}: {sample_avg:.2f} mins")


12. UNIQUE USERS SET & FEATURE DURATION FUNCTION
Unique users in set: 335
Average duration for 'Home Feed': 10.07 mins


In [15]:
# -------------------------------------------------------------------------
# Task 13: NumPy mean, median, and standard deviation of session duration
# -------------------------------------------------------------------------
print("\n" + "=" * 72)
print("13. NUMPY SESSION DURATION STATISTICS")
print("=" * 72)
durations_arr = df[duration_col].to_numpy(dtype=float)
np_mean = float(np.mean(durations_arr))
np_median = float(np.median(durations_arr))
np_std = float(np.std(durations_arr, ddof=1))  # sample standard deviation
print(f"Mean:               {np_mean:.2f} mins")
print(f"Median:             {np_median:.2f} mins")
print(f"Standard Deviation: {np_std:.2f} mins")


13. NUMPY SESSION DURATION STATISTICS
Mean:               11.04 mins
Median:             6.60 mins
Standard Deviation: 15.95 mins


In [16]:
# -------------------------------------------------------------------------
# Task 14: IQR outliers for session duration & Engagement Labels
# -------------------------------------------------------------------------
print("\n" + "=" * 72)
print("14. IQR OUTLIERS & ENGAGEMENT LABELS")
print("=" * 72)
q1, q3 = np.percentile(durations_arr, [25, 75])
iqr = float(q3 - q1)
upper_fence = float(q3 + 1.5 * iqr)
lower_fence = float(max(0.0, q1 - 1.5 * iqr))

df["Is_Long_Session_Outlier"] = df[duration_col] > upper_fence
long_outliers = df[df["Is_Long_Session_Outlier"]]
print(f"Q1 (25th pct): {q1:.2f} mins | Q3 (75th pct): {q3:.2f} mins | IQR: {iqr:.2f} mins")
print(f"Upper outlier fence (Q3 + 1.5*IQR): {upper_fence:.2f} mins (Lower fence: {lower_fence:.2f} mins)")
print(f"Unusually long sessions detected: {len(long_outliers)} ({len(long_outliers) / total_sessions * 100:.2f} %)")

df["Engagement_Level"] = df[duration_col].apply(lambda d: assign_engagement_label(d, q1, q3))
print("\nEngagement label distribution:")
print(df["Engagement_Level"].value_counts())


14. IQR OUTLIERS & ENGAGEMENT LABELS
Q1 (25th pct): 3.38 mins | Q3 (75th pct): 12.40 mins | IQR: 9.03 mins
Upper outlier fence (Q3 + 1.5*IQR): 25.94 mins (Lower fence: 0.00 mins)
Unusually long sessions detected: 100 (8.33 %)

Engagement label distribution:
Engagement_Level
Medium    601
Low       300
High      299
Name: count, dtype: int64


In [17]:
# -------------------------------------------------------------------------
# Task 15: AppUser class instances
# -------------------------------------------------------------------------
print("\n" + "=" * 72)
print("15. APPUSER OBJECTS (SAMPLE)")
print("=" * 72)
user_stats = (
    df.groupby(user_id_col, as_index=False)
    .agg(
        device_type=(device_col, lambda s: s.mode().iloc[0] if not s.mode().empty else "Unknown"),
        total_sessions=(session_id_col, "count"),
        avg_duration_mins=(duration_col, "mean"),
    )
)
app_users = [
    AppUser(
        user_id=str(row[user_id_col]),
        device_type=str(row["device_type"]),
        total_sessions=int(row["total_sessions"]),
        avg_duration_mins=float(row["avg_duration_mins"]),
    )
    for _, row in user_stats.head(5).iterrows()
]
for u in app_users:
    print(" -", u.summary())


15. APPUSER OBJECTS (SAMPLE)
 - AppUser(user_id='U0001', device_type='Android', sessions=3, avg_duration=7.93 mins)
 - AppUser(user_id='U0002', device_type='Android', sessions=6, avg_duration=20.38 mins)
 - AppUser(user_id='U0003', device_type='Android', sessions=1, avg_duration=13.80 mins)
 - AppUser(user_id='U0004', device_type='Android', sessions=2, avg_duration=9.60 mins)
 - AppUser(user_id='U0005', device_type='Web', sessions=1, avg_duration=10.80 mins)
